In [1]:
!pip install -q groq faiss-cpu sentence-transformers pandas tqdm transformers

In [2]:
import sys
import os

# Add project root to path so modules are importable
sys.path.insert(0, os.path.abspath(".."))

from config import client, embed_model, BASE_MODEL, JUDGE_MODEL
from data.loader import load_tatqa
from indexing.vector_store import build_vector_store
from pipeline import run_comparison
from evaluation.evaluator import evaluate_all, summarize_results

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ready!


In [4]:
# Load data — update path to your local TAT-QA file
DATA_PATH = "../data/tatqa_dataset_train.json"
samples = load_tatqa(DATA_PATH, max_samples=100)
print(f"Loaded {len(samples)} samples")

Loaded 100 samples


In [5]:
# Build baseline index (text only)
baseline_index, baseline_chunks, baseline_metadata = build_vector_store(
    samples, embed_model, use_table_aware_chunking=False
)

# Build improved index (text + table-aware chunking)
improved_index, improved_chunks, improved_metadata = build_vector_store(
    samples, embed_model, use_table_aware_chunking=True
)

Created 153 unique chunks from 100 samples
Embedding chunks...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

FAISS index built with 153 vectors
Created 766 unique chunks from 100 samples
Embedding chunks...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

FAISS index built with 766 vectors


In [6]:
# Run a single comparison on the first sample question
sample_question = samples[0]["question"]

t1, t2, t3 = run_comparison(
    question=sample_question,
    baseline_index=baseline_index,
    baseline_chunks=baseline_chunks,
    baseline_metadata=baseline_metadata,
    improved_index=improved_index,
    improved_chunks=improved_chunks,
    improved_metadata=improved_metadata,
    embed_model=embed_model,
    client=client,
    BASE_MODEL=BASE_MODEL,
    JUDGE_MODEL=JUDGE_MODEL,
    mode="improved",
    top_k=10,
)


QUESTION: What does the Weighted average actuarial assumptions consist of?
RETRIEVER MODE: IMPROVED

Retrieved 13 chunks:
  [e78f8b29-6085-43de-b32f-be1a68641be3_table_section_0] (table_section, score=1.315): Section: Weighted average actuarial assumptions used at 31 March1:...
  [e78f8b29-6085-43de-b32f-be1a68641be3_table_cell_11] (table_cell, score=1.297): Section: Weighted average actuarial assumptions used at 31 March1: | Row: Discount rate | Column: 2017 % | Value: 2.6...
  [e78f8b29-6085-43de-b32f-be1a68641be3_table_row_12] (table_row, score=1.295): Section: Weighted average actuarial assumptions used at 31 March1: | Row: Discount rate | Column: 2019 % | Value: 2.3 ; Column: 2018 % | Value: 2.5 ; Column: 20...
  [e78f8b29-6085-43de-b32f-be1a68641be3_table_cell_9] (table_cell, score=1.285): Section: Weighted average actuarial assumptions used at 31 March1: | Row: Discount rate | Column: 2019 % | Value: 2.3...
  [e78f8b29-6085-43de-b32f-be1a68641be3_table_cell_10] (table_cell, sco

In [7]:
# Evaluate baseline
baseline_df = evaluate_all(
    samples=samples,
    baseline_index=baseline_index,
    baseline_chunks=baseline_chunks,
    baseline_metadata=baseline_metadata,
    improved_index=improved_index,
    improved_chunks=improved_chunks,
    improved_metadata=improved_metadata,
    embed_model=embed_model,
    client=client,
    BASE_MODEL=BASE_MODEL,
    JUDGE_MODEL=JUDGE_MODEL,
    mode="baseline",
    max_samples=15,
    top_k=6,
)

# Evaluate improved
improved_df = evaluate_all(
    samples=samples,
    baseline_index=baseline_index,
    baseline_chunks=baseline_chunks,
    baseline_metadata=baseline_metadata,
    improved_index=improved_index,
    improved_chunks=improved_chunks,
    improved_metadata=improved_metadata,
    embed_model=embed_model,
    client=client,
    BASE_MODEL=BASE_MODEL,
    JUDGE_MODEL=JUDGE_MODEL,
    mode="improved",
    max_samples=15,
    top_k=6,
)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:14<00:00, 24.95s/it]


In [8]:
import pandas as pd

# Build and display comparison summary
baseline_summary = summarize_results(baseline_df, label="Baseline (text only)")
improved_summary = summarize_results(improved_df, label="Improved (table-aware)")

comparison_df = pd.concat([baseline_summary, improved_summary], ignore_index=True)
comparison_df

,system,tier1_f1,tier2_f1,tier3_f1,tier1_em,tier2_em,tier3_em,tier1_avg_tokens,tier2_avg_tokens,tier3_avg_tokens,tier1_avg_latency,tier2_avg_latency,tier3_avg_latency,tier3_debate_rate
0,Baseline (text only),0.051111,0.019471,0.051111,0.000000,0.000000,0.0,557.400000,1137.333333,603.2,4.294600,8.6558,4.3148,0.066667
1,Improved (table-aware),0.285545,0.324791,0.350823,0.133333,0.133333,0.2,1212.133333,2476.600000,1575.0,11.884267,24.5636,12.1064,0.266667
